# Evaluation

Generate all evaluation figures for the report.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, os, json, numpy as np, matplotlib.pyplot as plt, seaborn as sns, pandas as pd
sys.path.append('..')

print('Ready')

In [ ]:
# Load results
results = {}
result_dir = '../reports/logs'
if os.path.exists(result_dir):
    for fname in os.listdir(result_dir):
        if fname.endswith('.json'):
            with open(os.path.join(result_dir, fname)) as f:
                results[fname.replace('.json', '')] = json.load(f)
    print(f'Loaded {len(results)} result files')
    for k in results:
        print(f'  - {k}')
else:
    print('No results found. Run evaluations first.')

In [ ]:
# Gesture confusion matrix
if 'gesture_evaluation' in results:
    plt.figure(figsize=(8, 6))
    cm = np.array(results['gesture_evaluation']['confusion_matrix'])
    labels = [k for k in ['OPEN_PALM', 'FIST', 'THUMBS_UP', 'POINT', 'PEACE']
              if k in str(results['gesture_evaluation'].get('classification_report', ''))]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Gesture Classification')
    os.makedirs('../reports/figures', exist_ok=True)
    plt.savefig('../reports/figures/gesture_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Gesture accuracy: {results['gesture_evaluation']['accuracy']:.4f}")

In [ ]:
# Gaze confusion matrix
if 'gaze_evaluation' in results:
    plt.figure(figsize=(8, 6))
    cm = np.array(results['gaze_evaluation']['confusion_matrix'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['CENTER', 'LEFT', 'RIGHT', 'UP', 'DOWN'],
                yticklabels=['CENTER', 'LEFT', 'RIGHT', 'UP', 'DOWN'])
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Gaze Estimation')
    plt.savefig('../reports/figures/gaze_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Gaze accuracy: {results['gaze_evaluation']['accuracy']:.4f}")

In [ ]:
# System evaluation
if 'system_evaluation' in results:
    sysres = results['system_evaluation']
    print('=== System Performance ===')
    print(f"Average FPS: {sysres['fps']:.1f}")
    print(f"Face detection rate: {sysres['face_detection_rate']:.3f}")
    print(f"Centering error X: {sysres['mean_centering_error_x']:.4f}")
    print(f"Centering error Y: {sysres['mean_centering_error_y']:.4f}")

In [ ]:
# Summary table for report
summary = {}
for key, data in results.items():
    if 'accuracy' in data:
        summary[key] = data['accuracy']
        
pd.DataFrame([summary]) if summary else print('No accuracy metrics to summarize')